# 02 – Join Layers to SWORD
**Purpose:** Expand SWORD reaches with attributes from vector and raster dataset. Each section adds columns to a growing SWORD GeoDataFrame. The final output contains all joined attributes.

**To-Do:**

- [ ] sections einfügen, um zum SWORD hinzugefügte Datensätze mit dem original zu vergleichen
- 2.1.2 [ ] Probleme mit Strahler Ordnung, da diese entweder zu fein ist (z.B. von GRIT strahler bis z.B. 113 für Naryn), es muss eher sowas wie eine connected components analyse sein...mit NEXT_DOWN von RiverATLAS?
- 2.1.3 [] FFR match confidence lower than for RiverATLAS. 88 of 138 reaches are marked as ambigious (less than 60% of the sample points along the reach matched the same FFR feature)

---
---
## 2.0 | Imports

In [1]:
import geopandas as gpd
import os
import sys
import fiona
import webbrowser
import subprocess
import rasterio

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT, SAGA_CMD, SAGA_ENV, IN_BASIN_5, IN_SWORD as IN_SWORD_GLOBAL, CONTINENT
from _00_config import STUDY_AREA, PFAF_IDS, PFAF_LEVEL_DIGITS
from _00_setup import setup_study_area
from _2_1_1_line import join_line_majority, compute_strahler_segments
from _2_2_1_raster import (
    extract_raster_values, extract_raster_majority,
    compute_mrvbf, rasterize_sword,
    compute_vertical_distance, compute_horizontal_distance,
    reproject_dem_to_utm, clip_dem_to_aoi 
)
from _3_1_compute_features import compute_cross_sections, find_breakpoints, compute_floodplain_probability, compute_confinement_rinaldi

SAGA found: C:\Program Files\QGIS 3.42.3\apps\saga7\saga_cmd.exe


In [2]:
# ============================================================
# INPUT / OUTPUT OVERVIEW
# Study area is defined in _00_config.py
# ============================================================

# Load AOI and base SWORD from study area setup
aoi, sword_base = setup_study_area(
    basin_path        = IN_BASIN_5,
    sword_path        = IN_SWORD_GLOBAL,
    pfaf_ids          = PFAF_IDS,
    pfaf_level_digits = PFAF_LEVEL_DIGITS
)
bbox = tuple(aoi.total_bounds)

# Starting input for Section 2.1.1 is the clipped SWORD from Notebook 00
IN_SWORD = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_clip.gpkg")

# 2.1 LINE
#--- 2.1.1 grit v1.0 ------------------------------------------
IN_GRIT   = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example",
                          f"{STUDY_AREA}_gritv1.0.gpkg")
GRIT_LAYER = None
OUT_2_1_1  = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_1.gpkg")
OUT_MAP_2_1_1 = os.path.join(DATA_OUTPUT, "02_1_1_grit_map.html")

#--- 2.1.2 RiverATLAS ------------------------------------------
IN_RIVERATLAS = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example",
                              f"{STUDY_AREA}_riveratlas_v10.gpkg")
RIVERATLAS_COLS = [
    "ORD_STRA", "ORD_CLAS", "DIS_AV_CMS", "UPLAND_SKM",
    "slp_dg_cav", "ele_mt_cav", "sgr_dk_rav", "lit_cl_cmj",
    "HYRIV_ID", "NEXT_DOWN"
]
OUT_2_1_2     = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_2_riveratlas.gpkg")
OUT_MAP_2_1_2 = os.path.join(DATA_OUTPUT, "02_1_2_riveratlas_map.html")

#--- 2.1.3 FFR -------------------------------------------------
# FFR continent derived from CONTINENT (auto-detected from PFAF_IDS)
FFR_CONTINENT_MAP = {
    "as": "asia", "eu": "europe", "af": "africa",
    "na": "north_america", "sa": "south_america", "oc": "australia"
}
FFR_CONTINENT = FFR_CONTINENT_MAP[CONTINENT]

# Which cols should be joined
# Rename added columns in SWORD dataset
FFR_RENAME = {
    "REACH_ID": "reach_id_ffr",
    "BAS_ID":  "bas_id_ffr", 
    "GOID": "goid_ffr",
    
    "DOF": "dof_ffr",
    "DOR": "dor_ffr",
    "SED": "sed_ffr",
    "CSI": "csi_ffr",
    "CSI_FF": "csi_ff_ffr",
    "CSI_FF1": "csi_ff1_ffr",
    "CSI_FF2": "csi_ff2_ffr",
    
    "USE": "use_ffr",
    "RDD": "rdd_ffr",
    "URB": "urb_ffr",
    "FLD": "fld_ffr"
}
IN_FFR    = os.path.join(DATA_RAW, "0_data", "FFR", f"ffr_{FFR_CONTINENT}.gpkg")
OUT_2_1_3 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_3_ffr.gpkg")

#--- 2.1.4 GloRiC v1.0 ----------------------------------------
IN_GLORIC_V10 = os.path.join(DATA_RAW, "0_data", "GloRiC_v10.gpkg")
OUT_2_1_4     = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_4_gloric_v10.gpkg")

# 2.2 POINT
#--- 2.2.1 GDW barriers ----------------------------------------
IN_GDW    = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example",
                          f"{STUDY_AREA}_gdw_barriers_v1_0.gpkg")
OUT_2_2_1 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_2_1_gdw_snapped.gpkg")

# 2.4 RASTER
RASTER_JOINS = [
    ("STI_glowabio", os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example",
                                   "glowabio", f"{STUDY_AREA}_sti.tif"), "continuous"),
    ("worldcover",   os.path.join(DATA_RAW, "0_data", "worldcover",
                                   "worldcover_merged.tif"), "categorical"),
    ("clay", os.path.join(DATA_RAW, "0_data", "soilgrids", "clay_0-5cm_mean.tif"), "continuous"),
    ("sand", os.path.join(DATA_RAW, "0_data", "soilgrids", "sand_0-5cm_mean.tif"), "continuous"),
    ("silt", os.path.join(DATA_RAW, "0_data", "soilgrids", "silt_0-5cm_mean.tif"), "continuous"),
    ("cfvo", os.path.join(DATA_RAW, "0_data", "soilgrids", "cfvo_0-5cm_mean.tif"), "continuous"),
]
OUT_2_4_1 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_4_1.gpkg")

# SAGA outputs
OUT_DIR_SAGA = os.path.join(DATA_PROCESSED, "saga_outputs", STUDY_AREA)
OUT_COP30_DEM = os.path.join(DATA_RAW, f"{STUDY_AREA}_example",
                               f"{STUDY_AREA}_cop30_dem.tif")

# Final output
OUT_FINAL = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_joined_final.gpkg")
LATEST    = IN_SWORD

print(f"Study area : {STUDY_AREA} | Continent: {CONTINENT}")
print(f"bbox       : {bbox}")
print(f"IN_SWORD   : {IN_SWORD}")

Loading HydroBASINS Level 5 from: D:\0_InnoLab\0_data\BasinATLAS_HydroSHEDS\BasinATLAS_v10_lev05.gpkg
Basin polygons found: 1 for PFAF_IDs: [46219]
AOI bbox: [71.3625     40.4375     78.35416667 42.46666667]

Loading SWORD from: D:\0_InnoLab\0_data\SWOT_river_database_SWORD\as_sword_reaches_v17b.gpkg
SWORD reaches in bbox: 214
SWORD reaches after PFAF_ID filter: 140
Study area : naryn | Continent: as
bbox       : (np.float64(71.36249999957539), np.float64(40.43750000022487), np.float64(78.35416666695818), np.float64(42.46666666673332))
IN_SWORD   : C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_clip.gpkg


In [ ]:
# Vertify all inputs exists:

check = [
    ("SWORD clipped", IN_SWORD),
    ("GRIT",IN_GRIT),
    ("GDW",IN_GDW),
    ("RiverATLAS", IN_RIVERATLAS),
] + [(f"Raster: {name} ({method})", path) for name, path, method in RASTER_JOINS]

all_ok = True
for label, path in check:
    exists = os.path.exists(path)
    status = "Found" if exists else "NOT FOUND: check path"
    print(f"{label:30s}: {status}")
    if not exists:
        all_ok = False

print()
print("All files found." if all_ok else "Fix missing paths.")

---
---
## 2.1 | Line

---
### 2.1.1 | Line Join – GRIT strahler order

Joins GRITv1.0 strahler order to each SWORD reach using majority-vote sampling. Sample points are placed every 50m along each reach. Each point is assigned to the nearest GRIT feature. The strahler order with the most votes wins.


<u>References:</u>

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025) URL: https://aquaknow.jrc.ec.europa.eu/dataset/global-river-topology-grit

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025). Global River Topology (GRIT): A bifurcating river hydrography. Water Resources Research, 61, e2024WR038308. https://doi.org/10.1029/2024WR038308


In [ ]:
LATEST = IN_SWORD

# Load Data:
sword = gpd.read_file(IN_SWORD)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

grit_layers = fiona.listlayers(IN_GRIT)
grit_layer = GRIT_LAYER if GRIT_LAYER else grit_layers[0]
grit_raw = gpd.read_file(IN_GRIT, layer=grit_layer)
print(f"GRIT loaded: {len(grit_raw)} features | CRS: {grit_raw.crs}")
print(f"GRIT columns: {grit_raw.columns.tolist()}")

Configuration Cell:

In [ ]:
# Configuration:
# Which cols should be joined
GRIT_COLS_TO_JOIN = ["strahler_order", "global_id", "geometry"]
# Rename added columns in SWORD dataset
GRIT_RENAME= {
    "strahler_order" : "strahler_order_GRITv1.0",
    "global_id" : "global_id_GRITv1.0"
}
MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare GRIT: keep only needed columns
missing = [c for c in GRIT_COLS_TO_JOIN
           if c != "geometry" and c not in grit_raw.columns]
if missing:
    print(f"Columns not found in GRIT: {missing}")
else:
    print("All requested GRIT columns found.")

cols_keep = [c for c in GRIT_COLS_TO_JOIN if c in grit_raw.columns]
grit = grit_raw[cols_keep].copy()

In [ ]:
# Function from "_2_1_1_line.py"
sword_2_1_1 = join_line_majority(
    sword= sword,
    grit= grit,
    cols_to_join = GRIT_COLS_TO_JOIN,
    rename_cols= GRIT_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M,
    prefix="grit"
)

# Convert to integer
for col in ["strahler_order_GRITv1.0", "global_id_GRITv1.0"]:
    if col in sword_2_1_1.columns:
        sword_2_1_1[col] = sword_2_1_1[col].astype("Int64")

print("\nNew columns added:")
new_cols = ["strahler_order_GRITv1.0", "global_id_GRITv1.0", "grit_majority_confidence"]
print(sword_2_1_1[["reach_id"] + new_cols].head(10))

In [ ]:
# Quality check:
print("Majority confidence score [0-1]:")
print(sword_2_1_1["grit_majority_confidence"].describe().round(3))

print(f"Ambiguous reaches (confidence < 0.6 or NaN): {sword_2_1_1['grit_ambiguous'].sum()}")

print("\nReaches per strahler_order_GRITv1.0:")
print(sword_2_1_1["strahler_order_GRITv1.0"].value_counts().sort_index())

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_1.to_file(OUT_2_1_1, driver="GPKG")
print(f"\nSaved: {OUT_2_1_1}")

LATEST = OUT_2_1_1

In [ ]:
# Interactive Map to check the joined strahler order information:
m = sword_2_1_1.explore(
    column="reach_id",
    cmap="turbo",
    tooltip=["reach_id", "river_name", "strahler_order_GRITv1.0",
             "strm_order", "grit_majority_confidence", "global_id_GRITv1.0"],
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_2_1_1)
webbrowser.open(OUT_MAP_2_1_1)
print(f"Map saved: {OUT_MAP_2_1_1}")

### 2.1.2 | RiverATLAS

Adding strahler order and some other variables from RiverATLAS v.1.0 instead of GRIT...because propably better usage for the egg.

`strahler_order_RiverATLAS` is a single attribute (e.g. the value 3) describing 
how large/downstream a reach is. It does NOT identify which reaches belong to 
the same physical river segment.

Problem: two completely unconnected streams in different parts of the same 
basin can both have Strahler order 3. Grouping reaches purely by 
`strahler_order` would incorrectly merge them into one Egg, even though they 
are spatially unrelated.

Solution: `compute_strahler_segments()` uses the RiverATLAS `NEXT_DOWN` 
topology (which reach flows into which) to identify groups of reaches that 
are BOTH of equal Strahler order AND directly connected to each other. 
This produces a new column, `strahler_segment_id`, which encodes spatial 
connectivity together with Strahler order, giving each physically connected 
river segment a unique group ID suitable for Egg construction.

Note: this step is specific to building spatially valid Egg groups. It is NOT 
needed for simple attribute joins (e.g. FFR connectivity indicators), where 
each column is just an additional reach-level attribute with no grouping 
purpose.

But need to check out which Egg Dimension is the best, cause the ID usage of SWORD dataset (reach_id) also works, but the Eggs are bigger than with the RiverATLAS approach.

In [ ]:
# Load Data:
# Load Data:
sword = gpd.read_file(OUT_2_1_1)  # load SWORD with GRIT join from previous section
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

riveratlas_raw = gpd.read_file(IN_RIVERATLAS)
print(f"RiverATLAS loaded: {len(riveratlas_raw)} features | CRS: {riveratlas_raw.crs}")
print(f"RiverATLAS columns: {riveratlas_raw.columns.tolist()}")

# Configuration:
# Which cols should be joined
# Rename added columns in SWORD dataset
GRIT_RENAME = {
    "ORD_STRA": "strahler_order_RiverATLAS",
    "ORD_CLAS": "shreve_stream_order_RiverATLAS",   # Shreve stream order
    "DIS_AV_CMS": "dis_RiverATLAS", # mean annual discharge (m³/s)
    "UPLAND_SKM": "upland_skm_RiverATLAS",  # upstream catchment area (km²)
    "slp_dg_cav": "slp_dg_cav_RiverATLAS",  # mean catchment slope (degrees)
    "ele_mt_cav": "ele_mt_cav_RiverATLAS",  # mean catchment elevation (m)
    "sgr_dk_rav": "stream_gradient_RiverATLAS",  # stream gradient (m/km)
    "lit_cl_cmj": "dom_lithology_RiverATLAS",
    "HYRIV_ID"  : "HYRIV_ID_RiverATLAS",  # dominant lithology class
    "strahler_segment_id": "strahler_segment_id_RiverATLAS",
    "NEXT_DOWN": "NEXT_DOWN_RiverATLAS",
}


# Compute spatially connected Strahler segments from RiverATLAS topology
riveratlas_segmented = compute_strahler_segments(riveratlas_raw)

# Add strahler_segment_id to RIVERATLAS_COLS so it gets joined to SWORD
RIVERATLAS_COLS_WITH_SEG = RIVERATLAS_COLS + ["strahler_segment_id"]

MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare GRIT: keep only needed columns
missing = [c for c in RIVERATLAS_COLS
           if c != "geometry" and c not in riveratlas_segmented.columns]
if missing:
    print(f"Columns not found in RiverATLAS: {missing}")
else:
    print("All requested GRIT columns found.")

# Function from "_2_1_1_line.py"
sword_2_1_2 = join_line_majority(
    sword= sword,
    grit= riveratlas_segmented[RIVERATLAS_COLS_WITH_SEG + ["geometry"]],
    cols_to_join = RIVERATLAS_COLS_WITH_SEG,
    rename_cols= GRIT_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M,
    prefix="RiverATLAS"
)

# Integer columns:
for col in ["strahler_order_RiverATLAS",
            "shreve_stream_order_RiverATLAS",
            "dom_lithology_RiverATLAS"]:
    if col in sword_2_1_2.columns:
        sword_2_1_2[col] = sword_2_1_2[col].astype("Int64")

# Float columns – keep as float, just round for readability:
for col in ["dis_RiverATLAS", "upland_skm_RiverATLAS",
            "slp_dg_cav_RiverATLAS", "ele_mt_cav_RiverATLAS",
            "stream_gradient_RiverATLAS"]:
    if col in sword_2_1_2.columns:
        sword_2_1_2[col] = sword_2_1_2[col].round(3)

print("\nNew columns added:")
new_cols = ["strahler_order_RiverATLAS",
    "shreve_stream_order_RiverATLAS",
    "dis_RiverATLAS",
    "upland_skm_RiverATLAS",
    "slp_dg_cav_RiverATLAS",
    "ele_mt_cav_RiverATLAS",
    "stream_gradient_RiverATLAS",
    "dom_lithology_RiverATLAS",
    "HYRIV_ID_RiverATLAS", 
    "RiverATLAS_majority_confidence",
    ]
print(sword_2_1_2[["reach_id"] + new_cols].head(10))

# Quality check:
print("\nUnique HYRIV_ID groups (= potential eggs):")
print(sword_2_1_2["HYRIV_ID_RiverATLAS"].nunique())
print(f"Avg SWORD reaches per HYRIV_ID: {len(sword_2_1_2) / sword_2_1_2['HYRIV_ID_RiverATLAS'].nunique():.1f}")

print("Majority confidence score [0-1]:")
print(sword_2_1_2["RiverATLAS_majority_confidence"].describe().round(3))

print(f"Ambiguous reaches (confidence < 0.6 or NaN): {sword_2_1_2['RiverATLAS_ambiguous'].sum()}")

print("\nReaches per strahler_order_RiverATLAS:")
print(sword_2_1_2["strahler_order_RiverATLAS"].value_counts().sort_index())

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_2.to_file(OUT_2_1_2, driver="GPKG")
print(f"\nSaved: {OUT_2_1_2}")

LATEST = OUT_2_1_2

# Interactive Map to check the joined strahler order information:
m = sword_2_1_2.explore(
    column="reach_id",
    cmap="turbo",
    tooltip=["reach_id", "river_name", "strahler_order_RiverATLAS","RiverATLAS_majority_confidence"],
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_2_1_2)
webbrowser.open(OUT_MAP_2_1_2)
print(f"Map saved: {OUT_MAP_2_1_2}")

---
### 2.1.3 | FreeFlowingRivers (FFR)

Grill, G., Lehner, B., Thieme, M., Geenen, B., Tickner, D., Antonelli, F., Babu, S., Borrelli, P.,
Cheng, L., Crochetiere, H., Ehalt Macedo, H., Filgueiras, R., Goichot, M., Higgins, J., Hogan, Z.,
Lip, B., McClain, M., Meng, J.., Mulligan, M., Nilsson, C., Olden, J.D., Opperman, J., Petry, P.,
Reidy Liermann, C., Saenz, L., Salinas-Rodríguez, S., Schelle, P., Schmitt, R.J.P., Snider, J., Tan,
F., Tockner, K., Valdujo, P.H., van Soesbergen, A., Zarfl, C. (2019): (Nature) Mapping the world's free-flowing rivers: datasets and materials. figshare
https://doi.org/10.6084/m9.figshare.7688801 


Country Boundaries based on HydroBASIN

**Structure Stuff:**

- REACH_ID: Reach Identifier. The reach identifier can be used to link this dataset to the HydroATLAS database.
- GOID: Global Object Identifier. 
- NOID: Network Object Identifier. Same purpose as GOID
- NUOID: Identifies the NOIDs of the next upstream river reach. If there is 'NoData' given, the reach a headwater reach. Otherwise, the field holds 2 or more NOIDs. In the case of multiple NOIDs, the NOIDs are separated by an underscore.
- NDOID: Identifies the NOID of the next downstream river reach.
- CON_ID &  CONTINENT
- BAS_ID: Basin ID based on HydroSHEDS

**Connectivity Stuff:**

- DOF: Degree of Fragmentation (0-100%)
- DOR: Degree of Regulation (0-100%)
- SED: Sediment Trapping Index (0-100%)
- CSI: Connectivity Status Index (0-100%)
- CSI_FFID: River stretch identifier. Additional identifier to distinguish contiguous river stretches.
- CSI_FF: CSI above or below free-flowing threshold. Indicates if the CSI value of a river reach is
below (value = 0) or above (value = 1) the threshold of 95%. The attribute is used to calculate the free-flowing status of the river (see CSI_FF1 and CSI_FF2).
- CSI_FF1: Free-flowing status (two categories). Indicates river reaches that belong to a river with “free-flowing” status (value = 1) or “non-free-flowing” status (value = 3). Note that the value 2 is
reserved for river stretches with “good connectivity” status (see CSI_FF2).
- CSI_FF2: Free-flowing status (three categories). Indicates
river reaches that belong to a river with "free-flowing” status (value = 1), or a river stretch
with "good connectivity" status (value = 2) or a river or river stretch with "impacted" status
(value = 3).

**Pressure indicator:**
- USE: Water consumption (0-100%)
- RDD: Road construction (0-100%)
- URB: Urbanization Pressure (0-100%)
- FLD: Inundation (floodplain) extent in river reach catchment (%).

In [ ]:
sword = gpd.read_file(OUT_2_1_2)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

ffr_raw = gpd.read_file(IN_FFR, bbox=bbox)
print(f"FFR loaded: {len(ffr_raw)} features | CRS: {ffr_raw.crs}")

In [ ]:
FFR_COLS = list(FFR_RENAME.keys())

MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare FFR: keep only needed columns + geometry
cols_keep = [c for c in FFR_COLS if c in ffr_raw.columns]
cols_keep = cols_keep + ["geometry"]
ffr = ffr_raw[cols_keep].copy()

# Join FFR attributes to SWORD via majority-vote sampling
sword_2_1_3 = join_line_majority(
    sword=sword,
    grit=ffr,
    cols_to_join=FFR_COLS,
    rename_cols=FFR_RENAME,
    max_distance_m=MAX_DISTANCE_M,
    sample_distance_m=SAMPLE_DISTANCE_M,
    prefix="FFR"
)

NOTE: FFR match confidence is notably lower than RiverATLAS (mean ~0.55 vs RiverATLAS's higher confidence). This likely reflects different reach segmentation between FFR and SWORD:

In [ ]:
# Checking Type of original FFR columns
print(ffr_raw[["DOF", "DOR", "SED", "CSI", "CSI_FF", "GOID", "BAS_ID"]].dtypes)
print(ffr_raw[["DOF", "DOR", "SED", "CSI"]].describe())

print(sword_2_1_3["FFR_majority_confidence"].describe())
print(f"Ambiguous: {sword_2_1_3['FFR_ambiguous'].sum()}")

In [ ]:
# Integer columns (IDs and categorical):
for col in ["goid_ffr", "bas_id_ffr", "csi_ff_ffr", "csi_ff1_ffr", "csi_ff2_ffr"]:
    if col in sword_2_1_3.columns:
        sword_2_1_3[col] = sword_2_1_3[col].astype("Int64")

# Float columns (continuous percentage values), rounded for readability:
for col in ["dof_ffr", "dor_ffr", "sed_ffr", "csi_ffr", "use_ffr", "rdd_ffr", "urb_ffr", "fld_ffr"]:
    if col in sword_2_1_3.columns:
        sword_2_1_3[col] = sword_2_1_3[col].round(3)

In [ ]:
sword_2_1_3.to_file(OUT_2_1_3, driver="GPKG")
print(f"\nSaved: {OUT_2_1_3}")

LATEST = OUT_2_1_3

In [ ]:
print("SWORD reach length (reach_len):")
print(sword["reach_len"].describe())

print("\nFFR reach length (LENGTH_KM, in km - convert to m):")
ffr_naryn = ffr_raw  # already clipped/filtered to relevant area if applicable
print((ffr_naryn["LENGTH_KM"] * 1000).describe())

In [ ]:
# Pick one ambiguous reach
ambiguous_reach = sword_2_1_3[sword_2_1_3["FFR_ambiguous"] == True].iloc[0]
print(f"Reach ID: {ambiguous_reach['reach_id']}")
print(f"Confidence: {ambiguous_reach['FFR_majority_confidence']}")

### 2.1.4 | Global River Classification (GloRiC)

**Collumns which will be added:**

| Column | Description | 
| --- | --- |
|class_hydr|Hydrological sub-classes|
|Log_Q_avg|Long term discharge|
|Log_Q_var|Flow regime variability|
|CMI_indx |Climate moisture index|
|class_phys| Classes of physio-climatic sub-classification, see:|
|Lake_wet|Lake or wetland influence |
|stream_pow |Total stream power|
|Class_geom |Classes of geomorphic sub-classification (see....)|

---

`class_hydr`:
| Class number | Flow regime variability (1st digit) | Discharge average (2nd digit) |
|---:|---|---|
| 11 | low variability | very low discharge |
| 12 | low variability | low discharge |
| 13 | low variability | medium discharge |
| 14 | low variability | high discharge |
| 15 | low variability | very high discharge |
| 21 | medium variability | very low discharge |
| 22 | medium variability | low discharge |
| 23 | medium variability | medium discharge |
| 24 | medium variability | high discharge |
| 25 | medium variability | very high discharge |
| 31 | high variability | very low discharge |
| 32 | high variability | low discharge |
| 33 | high variability | medium discharge |
| 34 | high variability | high discharge |
| 35 | high variability | very high discharge |
|


`class_phys`:
| Class number | Minimum temperature (1st digit) | CMI (2nd digit) | Elevation (3rd digit) |
|---:|---|---|---|
| 111 | low temperature | low CMI | low elevation |
| 112 | low temperature | low CMI | high elevation |
| 121 | low temperature | medium CMI | low elevation |
| 122 | low temperature | medium CMI | high elevation |
| 131 | low temperature | high CMI | low elevation |
| 132 | low temperature | high CMI | high elevation |
| 211 | medium temperature | low CMI | low elevation |
| 212 | medium temperature | low CMI | high elevation |
| 221 | medium temperature | medium CMI | low elevation |
| 222 | medium temperature | medium CMI | high elevation |
| 231 | medium temperature | high CMI | low elevation |
| 232 | medium temperature | high CMI | high elevation |
| 311 | high temperature | low CMI | low elevation |
| 312 | high temperature | low CMI | high elevation |
| 321 | high temperature | medium CMI | low elevation |
| 322 | high temperature | medium CMI | high elevation |
| 331 | high temperature | high CMI | low elevation |
| 332 | high temperature | high CMI | high elevation |
| 411 | very high temperature | low CMI | low elevation |
| 412 | very high temperature | low CMI | high elevation |
| 421 | very high temperature | medium CMI | low elevation |
| 422 | very high temperature | medium CMI | high elevation |
| 431 | very high temperature | high CMI | low elevation |
| 432 | very high temperature | high CMI | high elevation |
|


`Lake_wet`:
| Class number | Lake-wetland influence (1st digit) | Stream power (2nd digit) |
|---:|---|---|
| 11 | no lakes or wetlands | low stream power |
| 12 | no lakes or wetlands | high stream power |
| 21 | lake-wetland influenced | low stream power |
| 22 | lake-wetland influenced | high stream power |

`Class_geom`:
| Class number | Reduced physio-climatic class (1st digit) | Reduced hydrologic class (2nd digit) | Reduced geomorphic class (3rd digit) |
|---:|---|---|---|
| 111 | cold, low and medium moisture region | very small river | low stream power |
| 112 | cold, low and medium moisture region | very small river | medium and high stream power |
| 113 | cold, low and medium moisture region | very small river | lake-wetland influenced |
| 121 | cold, low and medium moisture region | small river | low stream power |
| 122 | cold, low and medium moisture region | small river | medium and high stream power |
| 123 | cold, low and medium moisture region | small river | lake-wetland influenced |
| 131 | cold, low and medium moisture region | medium river | low stream power |
| 132 | cold, low and medium moisture region | medium river | medium and high stream power |
| 133 | cold, low and medium moisture region | medium river | lake-wetland influenced |
| 141 | cold, low and medium moisture region | large river | low stream power |
| 142 | cold, low and medium moisture region | large river | medium and high stream power |
| 143 | cold, low and medium moisture region | large river | lake-wetland influenced |
| 150 | cold, low and medium moisture region | very large river | - |
| 211 | cold, high moisture region | very small river | low stream power |
| 212 | cold, high moisture region | very small river | medium and high stream power |
| 213 | cold, high moisture region | very small river | lake-wetland influenced |
| 221 | cold, high moisture region | small river | low stream power |
| 222 | cold, high moisture region | small river | medium and high stream power |
| 223 | cold, high moisture region | small river | lake-wetland influenced |
| 231 | cold, high moisture region | medium river | low stream power |
| 232 | cold, high moisture region | medium river | medium and high stream power |
| 233 | cold, high moisture region | medium river | lake-wetland influenced |
| 241 | cold, high moisture region | large river | low stream power |
| 242 | cold, high moisture region | large river | medium and high stream power |
| 243 | cold, high moisture region | large river | lake-wetland influenced |
| 250 | cold, high moisture region | very large river | - |
| 311 | warm and hot, low moisture region | very small river | low stream power |
| 312 | warm and hot, low moisture region | very small river | medium and high stream power |
| 313 | warm and hot, low moisture region | very small river | lake-wetland influenced |
| 321 | warm and hot, low moisture region | small river | low stream power |
| 322 | warm and hot, low moisture region | small river | medium and high stream power |
| 323 | warm and hot, low moisture region | small river | lake-wetland influenced |
| 331 | warm and hot, low moisture region | medium river | low stream power |
| 332 | warm and hot, low moisture region | medium river | medium and high stream power |
| 333 | warm and hot, low moisture region | medium river | lake-wetland influenced |
| 341 | warm and hot, low moisture region | large river | low stream power |
| 342 | warm and hot, low moisture region | large river | medium and high stream power |
| 343 | warm and hot, low moisture region | large river | lake-wetland influenced |
| 350 | warm and hot, low moisture region | very large river | - |
| 411 | warm, medium moisture region | very small river | low stream power |
| 412 | warm, medium moisture region | very small river | medium and high stream power |
| 413 | warm, medium moisture region | very small river | lake-wetland influenced |
| 421 | warm, medium moisture region | small river | low stream power |
| 422 | warm, medium moisture region | small river | medium and high stream power |
| 423 | warm, medium moisture region | small river | lake-wetland influenced |
| 431 | warm, medium moisture region | medium river | low stream power |
| 432 | warm, medium moisture region | medium river | medium and high stream power |
| 433 | warm, medium moisture region | medium river | lake-wetland influenced |
| 441 | warm, medium moisture region | large river | low stream power |
| 442 | warm, medium moisture region | large river | medium and high stream power |
| 443 | warm, medium moisture region | large river | lake-wetland influenced |
| 450 | warm, medium moisture region | very large river | - |
| 511 | warm, high moisture region | very small river | low stream power |
| 512 | warm, high moisture region | very small river | medium and high stream power |
| 513 | warm, high moisture region | very small river | lake-wetland influenced |
| 521 | warm, high moisture region | small river | low stream power |
| 522 | warm, high moisture region | small river | medium and high stream power |
| 523 | warm, high moisture region | small river | lake-wetland influenced |
| 531 | warm, high moisture region | medium river | low stream power |
| 532 | warm, high moisture region | medium river | medium and high stream power |
| 533 | warm, high moisture region | medium river | lake-wetland influenced |
| 541 | warm, high moisture region | large river | low stream power |
| 542 | warm, high moisture region | large river | medium and high stream power |
| 543 | warm, high moisture region | large river | lake-wetland influenced |
| 550 | warm, high moisture region | very large river | - |
| 611 | hot, high moisture region | very small river | low stream power |
| 612 | hot, high moisture region | very small river | medium and high stream power |
| 613 | hot, high moisture region | very small river | lake-wetland influenced |
| 621 | hot, high moisture region | small river | low stream power |
| 622 | hot, high moisture region | small river | medium and high stream power |
| 623 | hot, high moisture region | small river | lake-wetland influenced |
| 631 | hot, high moisture region | medium river | low stream power |
| 632 | hot, high moisture region | medium river | medium and high stream power |
| 633 | hot, high moisture region | medium river | lake-wetland influenced |
| 641 | hot, high moisture region | large river | low stream power |
| 642 | hot, high moisture region | large river | medium and high stream power |
| 643 | hot, high moisture region | large river | lake-wetland influenced |
| 650 | hot, high moisture region | very large river | - |
| 711 | very hot, low moisture region | very small river | low stream power |
| 712 | very hot, low moisture region | very small river | medium and high stream power |
| 713 | very hot, low moisture region | very small river | lake-wetland influenced |
| 721 | very hot, low moisture region | small river | low stream power |
| 722 | very hot, low moisture region | small river | medium and high stream power |
| 723 | very hot, low moisture region | small river | lake-wetland influenced |
| 731 | very hot, low moisture region | medium river | low stream power |
| 732 | very hot, low moisture region | medium river | medium and high stream power |
| 733 | very hot, low moisture region | medium river | lake-wetland influenced |
| 741 | very hot, low moisture region | large river | low stream power |
| 742 | very hot, low moisture region | large river | medium and high stream power |
| 743 | very hot, low moisture region | large river | lake-wetland influenced |
| 750 | very hot, low moisture region | very large river | - |
| 811 | very hot, high moisture region | very small river | low stream power |
| 812 | very hot, high moisture region | very small river | medium and high stream power |
| 813 | very hot, high moisture region | very small river | lake-wetland influenced |
| 821 | very hot, high moisture region | small river | low stream power |
| 822 | very hot, high moisture region | small river | medium and high stream power |
| 823 | very hot, high moisture region | small river | lake-wetland influenced |
| 831 | very hot, high moisture region | medium river | low stream power |
| 832 | very hot, high moisture region | medium river | medium and high stream power |
| 833 | very hot, high moisture region | medium river | lake-wetland influenced |
| 841 | very hot, high moisture region | large river | low stream power |
| 842 | very hot, high moisture region | large river | medium and high stream power |
| 843 | very hot, high moisture region | large river | lake-wetland influenced |
| 850 | very hot, high moisture region | very large river | - |
| 911 | cold and warm, high elevation region | very small river | low stream power |
| 912 | cold and warm, high elevation region | very small river | medium and high stream power |
| 913 | cold and warm, high elevation region | very small river | lake-wetland influenced |
| 921 | cold and warm, high elevation region | small river | low stream power |
| 922 | cold and warm, high elevation region | small river | medium and high stream power |
| 923 | cold and warm, high elevation region | small river | lake-wetland influenced |
| 931 | cold and warm, high elevation region | medium river | low stream power |
| 932 | cold and warm, high elevation region | medium river | medium and high stream power |
| 933 | cold and warm, high elevation region | medium river | lake-wetland influenced |
| 941 | cold and warm, high elevation region | large river | low stream power |
| 942 | cold and warm, high elevation region | large river | medium and high stream power |
| 1011 | hot and very hot, high elevation region | very small river | low stream power |
| 1012 | hot and very hot, high elevation region | very small river | medium and high stream power |
| 1013 | hot and very hot, high elevation region | very small river | lake-wetland influenced |
| 1021 | hot and very hot, high elevation region | small river | low stream power |
| 1022 | hot and very hot, high elevation region | small river | medium and high stream power |
| 1023 | hot and very hot, high elevation region | small river | lake-wetland influenced |
| 1031 | hot and very hot, high elevation region | medium river | low stream power |
| 1032 | hot and very hot, high elevation region | medium river | medium and high stream power |
| 1033 | hot and very hot, high elevation region | medium river | lake-wetland influenced |
| 1041 | hot and very hot, high elevation region | large river | low stream power |
| 1042 | hot and very hot, high elevation region | large river | medium and high stream power |
| 1043 | hot and very hot, high elevation region | large river | lake-wetland influenced |

In [ ]:
# Load SWORD from previous section output
sword = gpd.read_file(OUT_2_1_3)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

gloric_raw = gpd.read_file(IN_GLORIC_V10, bbox=bbox)
print(f"GloRiC loaded: {len(gloric_raw)} features | CRS: {gloric_raw.crs}")

# Check exact column names before configuration
print(f"\nColumns: {gloric_raw.columns.tolist()}")

In [ ]:
# ============================================================
# Section 2.1.4 – GloRiC v1.0 Line Join
# Joins hydrologic, physio-climatic and geomorphic river
# classification from GloRiC onto SWORD reaches via majority vote.
#
# GloRiC is based on HydroRIVERS network (~4km reaches) –
# same topology as RiverATLAS, different segmentation than SWORD.
# Majority-vote confidence may be lower than RiverATLAS join.
# ============================================================

# Columns to join from GloRiC
GLORIC_COLS = [
    "Log_Q_avg",   # log10 of long-term average discharge (m3/s)
    "Log_Q_var",   # flow variability index (max monthly Q / avg Q)
    "Class_hydr",  # hydrologic class 11-35 (variability + discharge)
    "CMI_indx",    # climate moisture index
    "Class_phys",  # physio-climatic class
    "Lake_wet",    # 
    "Stream_pow",  # stream power (discharge x gradient)
    "Class_geom",  # geomorphic class
    "Reach_type",  # combined river reach type (127 classes)
]

# Rename columns for clarity in SWORD dataset
GLORIC_RENAME = {
    "Log_Q_avg"  : "log_q_avg_gloric",
    "Log_Q_var"  : "log_q_var_gloric",
    "Class_hydr" : "class_hydr_gloric",
    "CMI_indx"   : "cmi_indx_gloric",
    "Class_phys" : "class_phys_gloric",
    "Lake_wet"   : "lake_wet_gloric",
    "Stream_pow" : "stream_pow_gloric",
    "Class_geom" : "class_geom_gloric",
    "Reach_type" : "reach_type_gloric",
}

MAX_DISTANCE_M  = 100
SAMPLE_DISTANCE_M = 50

# Prepare GloRiC: keep only needed columns + geometry
cols_keep = [c for c in GLORIC_COLS if c in gloric_raw.columns]
cols_keep = cols_keep + ["geometry"]
gloric = gloric_raw[cols_keep].copy()
print(f"GloRiC prepared: {len(gloric)} features | {len(cols_keep)-1} columns")

# Join GloRiC to SWORD via majority-vote sampling
sword_2_1_4 = join_line_majority(
    sword          = sword,
    grit           = gloric,
    cols_to_join   = GLORIC_COLS,
    rename_cols    = GLORIC_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M,
    prefix         = "GloRiC"
)

In [ ]:
print("Class_hydr distribution after join:")
print(sword_2_1_4["class_hydr_gloric"].value_counts().sort_index())

print("\nClass_geom distribution:")
print(sword_2_1_4["class_geom_gloric"].value_counts().sort_index())

print("\nReach_type distribution:")
print(sword_2_1_4["reach_type_gloric"].value_counts().sort_index())

In [ ]:
# Integer columns (classification codes):
for col in ["class_hydr_gloric", "class_phys_gloric", 
            "class_geom_gloric", "reach_type_gloric"]:
    if col in sword_2_1_4.columns:
        sword_2_1_4[col] = sword_2_1_4[col].astype("Int64")

# Float columns (continuous values):
for col in ["log_q_avg_gloric", "log_q_var_gloric",
            "lake_wet_gloric","cmi_indx_gloric", "stream_pow_gloric"]:
    if col in sword_2_1_4.columns:
        sword_2_1_4[col] = sword_2_1_4[col].round(3)

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_4.to_file(OUT_2_1_4, driver="GPKG")
print(f"Saved: {OUT_2_1_4}")

LATEST = OUT_2_1_4

---
## 2.2 | Point Snap 

---
### 2.2.1 | GDW barrier

Snaps GDW barrier points to the nearest SWORD reach line. Only instream features are used. Snapped points are exported separately. The snap result (reach_id) is later used to flag reaches with upstream dams.

To-Do:
- [ ] Some of the original GDW points are located quite far from the feature, which is why the spatial deviation from the dam remains quite high even after snapping.

References:

Ward, B. (2019): "How to leverage Geopandas for faster snapping of points to lines". URL: https://medium.com/@brendan_ward/how-to-leverage-geopandas-for-faster-snapping-of-points-to-lines-6113c94e59aa

Lehner, B. et al.(2024): Global Dam Watch database version 1.0. URL: https://figshare.com/articles/dataset/Global_Dam_Watch_database_version_1_0/25988293

https://www.globaldamwatch.org/database

In [ ]:
# Reload SWORD from 2.1.1 output:
sword_2_1_1 = gpd.read_file(OUT_2_1_1)
print(f"SWORD reloaded: {len(sword_2_1_1)} reaches")

gdw = gpd.read_file(IN_GDW)
print(f"GDW loaded: {len(gdw)} features | CRS: {gdw.crs}")
print(f"GDW columns: {gdw.columns.tolist()}")

# Filtering for instream features only:
gdw = gdw[gdw["INSTREAM"].str.lower() == "instream"]
print(f"GDW instream: {len(gdw)} features after filter")

In [ ]:
# Configuration:
# The tolerance in meter describes the max snap distance
SNAP_TOLERANCE_M = 164

# NOTE: GDW points are exported separately, not merged into SWORD.
# The reach_id column links snapped points back to SWORD reaches.

In [ ]:
# Function from src file "_2_1_2_point.py"
snapped_gdw = snap_points_to_lines(
    points = gdw,
    lines = sword_2_1_1,
    tolerance_m = SNAP_TOLERANCE_M,
    line_id_col = "reach_id"
)

# Save snapped points (separate file -> not merged into SWORD)
snapped_gdw.to_file(OUT_2_2_1, driver="GPKG")
print(f"Saved: {OUT_2_2_1}")

POINTS = OUT_2_2_1

---
## 2.3 | Polygon 

...

---
---
## 2.4 | Raster Data

---
### 2.4.1 | Raster Join: STI glowabio

Extracts raster pixel values along each SWORD reach. Buffer per reach = (width/2) + offset to account for SWORD positional uncertainty (~50–200m offset from actual river).


Since the reaches can be offset by up to 200 meters from the actual river, a buffer is set around the reaches,depending on the reach width (column: "width"), to extract pixels from raster data. 

To-Do:

- [ ] I don't like the buffer solution because, with the current logic, it could end up extracting an unnecessary number of incorrect pixels. Maybe I should use a flow direction grid to better specify the direction of the buffer?
- [ ] In "raster_join.py" add to function "def extract_raster_values" that only pixels which overlap the buffer with a min. area of.... =< 50%? are included in the mean/median calculation.

https://glowabio.org/

https://hydrography.org/hydrography90m/hydrography90m_layers/


In [ ]:
# Reload from latest vector output
sword_current = gpd.read_file(LATEST)
print(f"SWORD reloaded: {len(sword_current)} reaches")

sword_2_4_1 = sword_current.copy()


for col_name, raster_path, method in RASTER_JOINS:
    print(f"\nRaster: {raster_path}")
    print(f"Method: {method}")
    
    if method == "continuous":
        sword_2_4_1 = extract_raster_values(
            sword_2_4_1, raster_path, col_name, width_col="width"
        )
    elif method == "categorical":
        sword_2_4_1 = extract_raster_majority(
            sword_2_4_1, raster_path, col_name, width_col="width"
        )
    else:
        print(f"Unknown method '{method}' for {col_name} - skipping")

In [ ]:
# Inspecting results, build column list based on method
raster_cols = []
for name, _, method in RASTER_JOINS:
    if method == "continuous":
        raster_cols += [f"{name}_mean", f"{name}_median"]
    elif method == "categorical":
        raster_cols += [f"{name}_majority", f"{name}_majority_confidence"]

# Filter to only existing columns
raster_cols = [c for c in raster_cols if c in sword_2_4_1.columns]

print("Sample:")
print(sword_2_4_1[["reach_id", "width"] + raster_cols].head(10))

print("\nStatistics:")
print(sword_2_4_1[raster_cols].describe())

# Save
sword_2_4_1.to_file(OUT_2_4_1, driver="GPKG")
print(f"\nSaved: {OUT_2_4_1}")

LATEST = OUT_2_4_1

---
### 2.4.2 Computing floodplain width to estimate Rinaldi, M. (2016) confinement 

This section is based on Mariam Ansah´s R-Script for automated floodplain delineation.

The steps are:
- 1. Rasterize SWORD onto DEM grid
- 2. Compute Multiresolution Index of Valley Bottom Flatness (MRVBF) with SAGA-GIS Tool
- 3. Compute Vertical Distance with SAGA-GIS Tool
- 4. Compute Horizontal Distance with SAGA-GIS Tool

In [4]:
# ============================================================
# 2.2.2 | Floodplain Indicators (SAGA GIS)
# Computes MRVBF, Vertical Distance and Horizontal Distance
# from COP30 DEM as input for confinement calculation.
# ============================================================

# Load latest SWORD output
sword = gpd.read_file(LATEST)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

# --- DEM Preparation ---

# Step 0a: Reproject DEM from EPSG:4326 to UTM
# Required: SAGA morphometry tools need metric CRS
OUT_COP30_DEM_UTM      = OUT_COP30_DEM.replace(".tif", "_utm.tif")
OUT_COP30_DEM_UTM_CLIP = OUT_COP30_DEM.replace(".tif", "_utm_clip.tif")

dem_utm_path = reproject_dem_to_utm(
    dem_path   = OUT_COP30_DEM,
    out_path   = OUT_COP30_DEM_UTM,
    crs_meters = "EPSG:32643"
)

# Step 0b: Clip UTM DEM to AOI
# Required: reduces processing area and avoids large VD values
dem_utm_path = clip_dem_to_aoi(
    dem_path = dem_utm_path,
    aoi_gdf  = aoi,
    out_path = OUT_COP30_DEM_UTM_CLIP,
    buffer_m = 5000
)
print(f"DEM ready: {dem_utm_path}")

SWORD loaded: 122 reaches | CRS: EPSG:4326
Reprojected DEM already exists, skipping: D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem_utm.tif
Clipped DEM already exists, skipping: D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem_utm_clip.tif
DEM ready: D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem_utm_clip.tif


In [ ]:
# Step 1: Rasterize SWORD using clipped UTM DEM as reference grid
river_rast_path = rasterize_sword(
    sword_gdf = sword,
    dem_path  = dem_utm_path,   # ← UTM clipped DEM
    out_path  = os.path.join(OUT_DIR_SAGA, "sword_raster.tif")
)

# Step 2: Compute MRVBF
mrvbf_path = compute_mrvbf(
    dem_path = dem_utm_path,    # !!!UTM clipped DEM
    out_dir  = OUT_DIR_SAGA,
    saga_cmd = SAGA_CMD,
    saga_env = SAGA_ENV
)

In [ ]:
# Step 3: Compute Vertical Distance
vd_path = compute_vertical_distance(
    dem_path        = dem_utm_path,    # ← UTM clipped DEM
    river_rast_path = river_rast_path,
    out_dir         = OUT_DIR_SAGA,
    saga_cmd        = SAGA_CMD,
    saga_env        = SAGA_ENV
)

# Step 4: Compute Horizontal Distance
hd_path = compute_horizontal_distance(
    river_rast_path = river_rast_path,
    out_dir         = OUT_DIR_SAGA,
    saga_cmd        = SAGA_CMD,
    saga_env        = SAGA_ENV
)


In [ ]:
# Before compute_cross_sections(), deciding the width:
mean_width   = sword["width"].median()
half_width_m = max(500, mean_width * 20)
print(f"Mean SWORD width : {mean_width:.0f} m")
print(f"Half width       : {half_width_m:.0f} m")

# Step 5: Generate cross-sections
cs_df = compute_cross_sections(
    sword_gdf           = sword,
    dem_path            = dem_utm_path,
    vd_path             = vd_path,
    hd_path             = hd_path,
    mrvbf_path          = mrvbf_path,
    n_sections          = 10, #50
    half_width_m        = half_width_m,   # ← automatisch
    points_per_transect = 50, #300
    crs_meters          = "EPSG:32643"
)

In [ ]:
print(cs_df.head())

# Step 6: Find breakpoints
thresholds = find_breakpoints(cs_df)

In [ ]:
# Step 7: Compute floodplain probability map
OUT_FLOOD_PROB  = os.path.join(OUT_DIR_SAGA, "floodplain_probability.tif")
flood_prob_path = compute_floodplain_probability(
    vd_path    = vd_path,
    hd_path    = hd_path,
    mrvbf_path = mrvbf_path,
    thresholds = thresholds,
    out_path   = OUT_FLOOD_PROB
)
print(f"Floodplain probability: {flood_prob_path}")

In [ ]:
print(f"vd_max    : {thresholds['vd_max']:.2f} m")
print(f"hd_max    : {thresholds['hd_max']:.2f} m")
print(f"mrvbf_min : {thresholds['mrvbf_min']:.3f}")

In [ ]:
result = subprocess.run(
    [SAGA_CMD, "grid_tools", "10", "--help"],
    capture_output=True,
    text=True,
    env=SAGA_ENV
)
print(result.stdout[:800])

In [ ]:
# Step 8: Compute valley_width

---
### 2.4.3 | Compute Confinement

In [8]:
# Compute Rinaldi confinement from DEM slope (fixed threshold 10°)
sword = gpd.read_file(LATEST)
print(f"SWORD loaded: {len(sword)} reaches")

confinement_pct = compute_confinement_rinaldi(
    sword_gdf           = sword,
    dem_path            = dem_utm_path,
    slope_threshold_deg = 10.0,
    bank_search_m       = 50.0,
    sample_spacing_m    = 100.0,
    crs_meters          = "EPSG:32643"
)

# Add to SWORD
sword["confinement_pct_rinaldi"] = confinement_pct

# Classification distribution
bins   = [0, 10, 90, 100]
labels = ["unconfined", "partly confined", "confined"]
classes = pd.cut(sword["confinement_pct_rinaldi"], bins=bins, labels=labels)
print(classes.value_counts().sort_index())

# Save
OUT_2_2_2 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_2_2_confinement.gpkg")
sword.to_file(OUT_2_2_2, driver="GPKG")
print(f"Saved: {OUT_2_2_2}")
LATEST = OUT_2_2_2

SWORD loaded: 122 reaches
Slope threshold (fixed): 10.0°


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

---
---
## 2.5 | FINAL OUTPUT

Combining all joined attributes into one final SWORD GeoDataFrame.
This is the input for Notebook 03 (classification).

In [ ]:
print(f"OUT_2_1_1 exists: {os.path.exists(OUT_2_1_1)}")
print(f"OUT_2_1_2 exists: {os.path.exists(OUT_2_1_2)}")
print(f"OUT_2_1_3 exists: {os.path.exists(OUT_2_1_3)}")
print(f"OUT_2_1_4 exists: {os.path.exists(OUT_2_1_4)}")
print(f"OUT_2_4_1 exists: {os.path.exists(OUT_2_4_1)}")
print(f"LATEST: {LATEST}")

In [ ]:
# Load the latest raster-joined dataset as base
sword_final = gpd.read_file(LATEST)

print(f"Final SWORD dataset:")
print(f"Reaches : {len(sword_final)}")
print(f"Columns : {sword_final.columns.tolist()}")

# Save as 
sword_final.to_file(OUT_FINAL, driver="GPKG")
print(f"\nSaved final output: {OUT_FINAL}")

In [ ]:
# Investigating some stuff
tooltip_cols = [c for c in [
    "reach_id", "river_name", "strahler_order_GRITv1.0",
    "grit_majority_confidence", "RiverATLAS_majority_confidence"
] if c in sword_final.columns]

m = sword_final.explore(
    column="strahler_order_GRITv1.0",
    cmap="RdYlBu",
    tooltip=tooltip_cols,
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_FINAL)
webbrowser.open(OUT_MAP_FINAL)
print(f"Map saved: {OUT_MAP_FINAL}")

save everything as parquet for streamlit app

In [ ]:
# Load latest output (includes all joins: GRIT, RiverATLAS, FFR, GloRiC, Raster.....)
sword_final = gpd.read_file(LATEST)
print(f"Final SWORD dataset:")
print(f"  Reaches : {len(sword_final)}")
print(f"  Columns : {len(sword_final.columns)}")

# Save as GeoPackage (full dataset for Notebook 03)
sword_final.to_file(OUT_FINAL, driver="GPKG")
print(f"\nSaved: {OUT_FINAL}")

# Also export as Parquet for Streamlit app
import os
app_data_dir = os.path.join("..", "app", "data")
os.makedirs(app_data_dir, exist_ok=True)
parquet_path = os.path.join(app_data_dir, f"sword_{STUDY_AREA}_joined.parquet")
sword_final.to_parquet(parquet_path)
print(f"Saved app data: {parquet_path}")

LATEST = OUT_FINAL